# Lab 03 solution

In [ ]:
from pathlib import Path
import csv
import pandas as pd

DATA = Path("../../data")
OUT = Path("output")

In [ ]:
for f in sorted(DATA.glob("*.*")):
    print(f"{f.name:30} {f.stat().st_size / 1024:8.1f} KB")

In [ ]:
total_2023 = 0.0
with open(DATA / "trade_summary.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["year"] == "2023":
            total_2023 += float(row["exports_usd_m"])

print(f"Total 2023 exports: {total_2023:,.0f} USD m")

In [ ]:
# check
assert total_2023 > 10_000_000
print("Task 2 OK")

In [ ]:
trade = pd.read_csv(DATA / "trade_summary.csv")
tariffs = pd.read_csv(DATA / "tariffs_mfn.csv")
print(pd.ExcelFile(DATA / "countries.xlsx").sheet_names)
countries = pd.read_excel(DATA / "countries.xlsx", sheet_name="countries")

for name, df in [("trade", trade), ("tariffs", tariffs), ("countries", countries)]:
    print(name, df.shape)
    print(df.dtypes, end="\n\n")

In [ ]:
files_to_load = [
    "trade_summary.csv",
    "tariffs_mfn.csv",
    "countries.xlsx",
    "imf_indicators.csv",   # does not exist
    "README.md",            # unsupported type
]

def load_dataset(path):
    path = Path(path)
    try:
        if path.suffix == ".csv":
            return pd.read_csv(path)
        if path.suffix in (".xlsx", ".xls"):
            return pd.read_excel(path)
        print(f"Unsupported file type: {path.name}")
    except FileNotFoundError:
        print(f"File not found: {path}")
    return None

datasets = {}
for name in files_to_load:
    df = load_dataset(DATA / name)
    if df is not None:
        datasets[name] = df

In [ ]:
# check
assert set(datasets) == {"trade_summary.csv", "tariffs_mfn.csv", "countries.xlsx"}
print("Task 4 OK")

In [ ]:
for name, df in datasets.items():
    print(name)
    print(df.isna().sum(), end="\n\n")

`tariffs_mfn.csv` has missing tariff values in `avg_mfn_tariff_pct`. The other two files are complete. We will decide how to treat gaps in Module 07.

In [ ]:
OUT.mkdir(exist_ok=True)

africa = trade[(trade["region"] == "Africa") & (trade["year"] == 2023)]
africa.to_csv(OUT / "africa_2023.csv", index=False)
africa.to_excel(OUT / "africa_2023.xlsx", index=False)

with open(OUT / "load_report.txt", "w", encoding="utf-8") as f:
    for name, df in datasets.items():
        f.write(f"{name}: {len(df)} rows\n")
print((OUT / "load_report.txt").read_text())

In [ ]:
# check
assert (OUT / "africa_2023.csv").exists()
assert (OUT / "africa_2023.xlsx").exists()
assert "trade_summary.csv" in (OUT / "load_report.txt").read_text()
print("Task 6 OK")